<a href="https://colab.research.google.com/github/VickkiMars/bible-seq/blob/main/BibleProjectTest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 50.4 MB/s eta 0:00:00


# DATA

### **Cleaning**

In [3]:
!cp -r drive/MyDrive/Bible\ Project temp/

In [100]:
def get_leading_number(verse_string):
    """
    Extracts the integer value of leading digits from a verse string.
    Returns None if no leading digits are found.
    """
    match = re.match(r'^(\d+)', verse_string)
    if match:
        return int(match.group(1))
    return None

In [69]:
import fitz
import re

EFIK_PATH = "temp/Efik-All-Bible.pdf"
NIV_PATH = "temp/NIV-Bible.pdf"

In [70]:
import fitz  # PyMuPDF
import re

def get_all_verses(*pdf_paths):
    all_results = []
    verse_start_pattern = re.compile(r'^(\d+)\s*(.*)', re.MULTILINE)

    for path in pdf_paths:
        try:
            doc = fitz.open(path)
            file_verses = []
            current_verse = ""

            for page in doc:
                text = page.get_text()
                lines = text.splitlines()

                for line in lines:
                    line = line.strip()
                    if not line: continue

                    # Check if line starts with a verse number
                    match = verse_start_pattern.match(line)

                    if match:
                        # If we were already building a verse, save it before starting new one
                        if current_verse:
                            file_verses.append(current_verse)
                        current_verse = line
                    else:
                        # If it doesn't start with a digit, it's a continuation of the previous verse
                        if current_verse:
                            current_verse += " " + line

            # Append the very last verse of the file
            if current_verse:
                file_verses.append(current_verse)

            all_results.append(file_verses)
            doc.close()

        except Exception as e:
            print(f"Error loading {path}: {e}")
            all_results.append([]) # Keep the list index consistent even on failure

    return all_results



In [135]:
efik, english = get_all_verses(EFIK_PATH, NIV_PATH)

In [136]:
efik = [ef for ef in efik if "Bible Society of Nigeria" not in ef][83:]
english = english[17:]

In [137]:
len(efik), len(english)

(31349, 31151)

In [139]:
# def fix_verse_mismatches(lang):
#     verses = []
#     i = 0
#     n = len(lang)

#     while i < n - 1:
#         current_verse = lang[i]
#         next_verse = lang[i+1]

#         try:
#             upper_verse = lang[i+2]
#         except IndexError:
#             upper_verse = ""

#         current_leading_number = get_leading_number(current_verse)
#         upper_leading_number = get_leading_number(upper_verse)

#         if current_leading_number is not None and upper_leading_number == current_leading_number + 1:
#             verses.append(current_verse + " " + next_verse)
#             i += 2  # The index jump: skips processing the concatenated next_verse
#             continue

#         verses.append(current_verse)
#         i += 1

#     # Append the final verse if it wasn't consumed by the final jump
#     if i == n - 1:
#         verses.append(lang[-1])

#     return verses

In [140]:
def fix_verse_mismatches(lang):
    verses = []
    i = 0
    n = len(lang)

    while i < n - 1:
        current_verse = lang[i]
        next_verse = lang[i+1]

        try:
            upper_verse = lang[i+2]
        except IndexError:
            upper_verse = ""

        current_leading_number = get_leading_number(current_verse)
        upper_leading_number = get_leading_number(upper_verse)

        if current_leading_number is not None and upper_leading_number == current_leading_number + 1:
            verses.append(current_verse + " " + next_verse)
            i += 2  # The index jump: skips processing the concatenated next_verse
            continue

        verses.append(current_verse)
        i += 1

    # Append the final verse if it wasn't consumed by the final jump
    if i == n - 1:
        verses.append(lang[-1])

    return verses

In [141]:
#english[141:141+22] #Uncomment to see pre-fix state (verse 14-17)

In [142]:
efik, english = fix_verse_mismatches(efik), fix_verse_mismatches(english)

In [143]:
#english[141:141+32] #Uncomment to see post-fix state (same)

In [144]:
len(efik), len(english)

(31230, 31018)

In [145]:
def get_all_chapter_1s_efik(efik_list):
  chapter_1_indices = []
  for i, verse in enumerate(efik_list):
    # Check if the verse starts with '1' immediately followed by a letter, and has a length greater than 10 characters
    if re.match(r'^1[a-zA-Z]', verse) and len(verse) > 10:
      chapter_1_indices.append(i)
  return chapter_1_indices

In [146]:
efik_ids = get_all_chapter_1s_efik(efik)

In [147]:
efik_verse_gaps = [efik_ids[i+1] - efik_ids[i] for i in range(len(efik_ids)-1)]

In [148]:
import re
def get_all_chapter_1s_english(eng_list):
  chapter_1_indices = []
  for i in range(len(eng_list) - 1):
    current_verse = eng_list[i]
    next_verse = eng_list[i+1]

    try:
      upper_verse = eng_list[i+2]
    except IndexError as e:
      pass

    current_leading_number = get_leading_number(current_verse)
    next_leading_number = get_leading_number(next_verse)
    upper_leading_number = get_leading_number(upper_verse)

    # Chapter 1 is labelled 2
    if next_leading_number == 2 and upper_leading_number == 2 :
      chapter_1_indices.append(i+1)
      continue

    # Chapter 1 is labelled 1
    if next_leading_number == 2 and current_leading_number == 1:
      chapter_1_indices.append(i)

    # Chapter 1 is labelled any other number
    if next_leading_number == 2 and upper_leading_number > 2:
      chapter_1_indices.append(i)


  return chapter_1_indices

In [149]:
eng_ids = get_all_chapter_1s_english(english)

In [150]:
len(eng_ids)

1403

In [151]:
def remove_duplicates_preserve_order(input_list):
    unique_list = []
    seen_elements = set()
    for item in input_list:
        if item not in seen_elements:
            unique_list.append(item)
            seen_elements.add(item)
    return unique_list

eng_ids = remove_duplicates_preserve_order(eng_ids)
print("Duplicates removed from eng_ids while preserving order.")
print(f"New length of eng_ids: {len(eng_ids)}")

Duplicates removed from eng_ids while preserving order.
New length of eng_ids: 1189


In [152]:
eng_ids = [eng_ids[i] for i in range(len(eng_ids)-1) if eng_ids[i] != eng_ids[i+1]]

In [153]:
english_verse_gaps = [eng_ids[i+1] - eng_ids[i] for i in range(len(eng_ids)-1)]

In [154]:
efik_verse_gaps[:15]

[31, 25, 24, 26, 32, 22, 24, 22, 29, 32, 32, 20, 18, 24, 21]

In [155]:
english_verse_gaps[:15]

[30, 25, 24, 26, 32, 22, 24, 22, 29, 32, 32, 20, 18, 24, 21]

In [156]:
len(efik_verse_gaps)

1037

In [157]:
len(english_verse_gaps)

1187

In [158]:
mismatches = [id for id in range(1037) if efik_verse_gaps[id] != english_verse_gaps[id]]

efvg, evg = efik_verse_gaps, english_verse_gaps

In [159]:
efvg[0], evg[0]

(31, 30)

In [160]:
efik[0:32], english[0:32]

(['1KE eritɔŋɔ Abasi okobot enyɔŋ ye isɔŋ.',
  '2Ndien isɔŋ edi ikpikpu, ye ukpɔk; ekim onyuŋ odu ke enyɔŋ inyaŋ. Spirit Abasi onyuŋ ɔkubɔ ke enyɔŋ mmɔŋ.',
  '3Ekem Abasi ɔdɔhɔ ete, Yak uŋwana odu: ndien uŋwana odu.',
  '4Ndien Abasi okut uŋwana, ete ɔfɔn:',
  '5Abasi onyuŋ adianare uŋwana ye ekim. Ndien Abasi okot uŋwana ete, Uwemeyo, onyuŋ okot ekim ete, Okoneyo. Ndien mbubreyo ye usenubɔk edi akpa usen.',
  '6Ndien Abasi ɔdɔhɔ ete, Yak ikpa-enyɔŋ odu ke ufɔt mmɔŋ, yak onyuŋ adiaŋare mmɔŋ ye mmɔŋ.',
  '7Ndien Abasi anam ikpa-enyɔŋ, onyuŋ adianare mmɔŋ emi odude ke idak ikpa-enyɔŋ ye mmɔŋ emi odorode ke ikpa-enyɔŋ: ndien edi ntre. Ndien Abasi okot ikpa enyɔŋ. Ete, Heaven.',
  '8Ndien mbubreyo ye usenubok edi udiana akpa usen.',
  '9Ndien Abasi ete, Yak mmɔŋ ke idak enyɔŋ oboho ke ebiet kiet, yak edisat isɔŋ onyuŋ ɔwɔrɔ: ndien edi ntre.',
  '10Ndien Abasi okot edisat isɔŋ ete, Obot; onyun okot mmɔŋ emi obohode, ete Inyaŋ: ndien Abasi okut ete efɔn.',
  '11Ndien Abasi ete, Yak isɔŋ otib

In [126]:
mismatches

[0,
 50,
 83,
 90,
 117,
 140,
 147,
 153,
 187,
 191,
 192,
 193,
 194,
 195,
 196,
 197,
 198,
 199,
 200,
 201,
 202,
 203,
 204,
 205,
 206,
 207,
 208,
 209,
 210,
 211,
 212,
 213,
 214,
 215,
 216,
 217,
 218,
 219,
 220,
 221,
 222,
 223,
 225,
 226,
 227,
 228,
 229,
 230,
 231,
 232,
 233,
 234,
 235,
 236,
 237,
 238,
 239,
 240,
 241,
 242,
 243,
 244,
 245,
 246,
 247,
 248,
 249,
 250,
 251,
 252,
 253,
 254,
 255,
 256,
 257,
 258,
 259,
 260,
 261,
 262,
 263,
 264,
 265,
 266,
 267,
 268,
 269,
 270,
 271,
 272,
 273,
 274,
 275,
 276,
 277,
 278,
 279,
 280,
 281,
 282,
 283,
 284,
 285,
 286,
 287,
 288,
 289,
 290,
 291,
 292,
 293,
 294,
 295,
 296,
 297,
 298,
 299,
 300,
 301,
 302,
 303,
 304,
 305,
 306,
 307,
 308,
 309,
 310,
 311,
 312,
 313,
 314,
 315,
 316,
 317,
 318,
 319,
 320,
 321,
 322,
 323,
 324,
 325,
 326,
 327,
 328,
 329,
 330,
 331,
 332,
 333,
 334,
 335,
 336,
 337,
 338,
 339,
 340,
 341,
 342,
 343,
 344,
 345,
 346,
 348,
 349,
 350,
 35

In [28]:
efik[138+32]

'11Ke ɔyɔhɔ isua uwem Noah ikie itiokiet, ke udiana akpa ɔfiɔŋ, ke ɔyɔhɔ usen ɔfiɔŋ efureba, ke usen oro kpukpru idim inyaŋ ibom asiaha, enyuŋ eberede window ikpa- enyɔŋ.'

In [38]:
new_eng = cleaner_pipeline(english[141:141+32])

In [44]:
len(new_eng)

28